# 3.04 Pinecone (Serverless)

**Pinecone**은 관리형 벡터 DB. serverless 인덱스는 사용량 기반 과금이며, 리전·클라우드만 고르면 인프라 운영 부담이 없다.

## 학습 목표

- `pinecone.Pinecone(...).create_index(ServerlessSpec)` 로 인덱스 생성
- `PineconeVectorStore.from_documents` / `from_texts` 로 업서트
- MongoDB 스타일 메타데이터 필터 (`$eq`, `$in`, `$and`, `$or`)
- `namespace` 로 테넌트·버전 분리

## 언제 쓰나

- 관리형 SLA·멀티리전 필요
- 인프라 팀 없이 RAG 프로덕션 시작
- 수천만~수억 벡터 규모 스케일

## 3.04.1 환경 설정 / 서비스 연결

`.env`에 `PINECONE_API_KEY` 필요. serverless 생성에는 별도 환경 변수 없다 — 리전은 코드에서 `ServerlessSpec(cloud="aws", region="us-east-1")` 로 지정.

In [ ]:
# !pip install -U langchain-pinecone langchain-openai

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"
assert os.environ.get("PINECONE_API_KEY"), "PINECONE_API_KEY 필요 (https://app.pinecone.io)"

## 3.04.2 인덱스 생성

임베딩 차원 수(`text-embedding-3-small` → 1536)와 metric을 반드시 명시. 이미 존재하면 그냥 가져다 쓴다.

In [ ]:
from pinecone import Pinecone, ServerlessSpec

INDEX_NAME = os.environ.get("PINECONE_INDEX", "langchain-demo")
DIMENSION = 1536  # text-embedding-3-small

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

existing = {ix["name"] for ix in pc.list_indexes()}
if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print("created:", INDEX_NAME)
else:
    print("exists:", INDEX_NAME)

index = pc.Index(INDEX_NAME)
print("stats:", index.describe_index_stats())

## 3.04.3 기본 사용법

`PineconeVectorStore.from_documents(docs, embedding, index_name=..., namespace=...)` 로 업서트. `namespace`는 같은 인덱스 내에서 **논리적 파티션** 역할을 한다 (예: `"prod"`, `"dev"`, `"tenant-a"`).

In [ ]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = [
    Document(page_content="Pinecone은 관리형 벡터 데이터베이스이다.", metadata={"source": "pinecone", "tier": "managed"}),
    Document(page_content="Serverless 인덱스는 사용량 기반 과금이다.", metadata={"source": "pinecone", "tier": "managed"}),
    Document(page_content="FAISS는 로컬 실행용 라이브러리이다.", metadata={"source": "faiss", "tier": "local"}),
    Document(page_content="Qdrant는 자체 호스팅도 가능하다.", metadata={"source": "qdrant", "tier": "hybrid"}),
]

NAMESPACE = "demo"
store = PineconeVectorStore.from_documents(
    docs,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
)

# 업서트가 반영되기까지 약간의 지연 — 실운영에서는 polling 또는 재시도
import time; time.sleep(3)
for d in store.similarity_search("관리형 벡터 DB", k=3, namespace=NAMESPACE):
    print("-", d.metadata["source"], "|", d.page_content[:40])

## 3.04.4 메타데이터 필터링 — MongoDB 스타일

Pinecone 필터는 dict에 `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, `$and`, `$or` 를 쓴다. `filter=` 인자를 `similarity_search` 에 바로 넘긴다.

In [ ]:
hits = store.similarity_search(
    "로컬 실행",
    k=5,
    namespace=NAMESPACE,
    filter={"tier": {"$in": ["local", "hybrid"]}},
)
for d in hits:
    print("-", d.metadata, "|", d.page_content[:40])

## 3.04.5 점수 포함 · MMR

Pinecone `similarity_search_with_score` 의 점수는 `metric="cosine"` 인덱스에서 **코사인 유사도**(1.0이 동일)로 나온다. Chroma/FAISS의 거리 점수와 방향이 반대임에 유의.

In [ ]:
print("--- 유사도 점수 (1.0=최대) ---")
for doc, sim in store.similarity_search_with_score("관리형 벡터 DB", k=3, namespace=NAMESPACE):
    print(f"{sim:.4f}  {doc.metadata['source']}")

print("\n--- MMR ---")
for d in store.max_marginal_relevance_search("벡터 검색 엔진", k=3, fetch_k=4, lambda_mult=0.4, namespace=NAMESPACE):
    print("-", d.metadata["source"])

## 3.04.6 영속성 · 재연결 · 정리

네트워크 DB이므로 자동 영속화. 동일한 `index_name` + `namespace` 로 `PineconeVectorStore(...)` 다시 만들면 즉시 기존 벡터 사용 가능. 실습 후 네임스페이스 삭제.

In [ ]:
reopened = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    namespace=NAMESPACE,
)
for d in reopened.similarity_search("serverless", k=2):
    print("-", d.metadata["source"], "|", d.page_content[:40])

# 실습 데이터 정리: namespace 전체 삭제
index.delete(delete_all=True, namespace=NAMESPACE)
print("namespace purged:", NAMESPACE)

## 체크리스트

- [ ] 인덱스 `dimension`은 임베딩 모델과 정확히 일치 — `text-embedding-3-small`=1536, `-large`=3072
- [ ] `metric`은 임베딩 모델 권장값(OpenAI는 cosine)을 따른다
- [ ] 테넌트/환경 분리는 `namespace`로 — 인덱스 여러 개보다 저렴하고 빠르다
- [ ] 업서트 직후 검색은 **eventual consistency** — 짧은 대기(수 초) 필요
- [ ] serverless 과금은 read/write 유닛과 저장 벡터 수 — 대량 삭제로 정리

## 다음

- `05_qdrant.ipynb` — 셀프 호스팅 가능한 Rust 엔진
- `05_retrievers/` — retriever 레이어와 결합

## 참고

- Pinecone docs: https://docs.pinecone.io/
- `langchain-pinecone`: https://python.langchain.com/docs/integrations/vectorstores/pinecone